In [2]:
!pip install -q pymupdf groq transformers peft trl bitsandbytes accelerate datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 72.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.9 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have nu

In [1]:
import json
import random

metadata_path = "/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json"

NUM_PAPERS = 15

TOPIC_KEYWORDS = [
    "single-cell", "single cell", "scrna", "scRNA",
    "transcriptomics", "cell atlas", "cell type", "rna sequencing"
]

candidate_papers = []

with open(metadata_path, "r") as f:
    for line in f:
        paper = json.loads(line)

        categories = paper.get("categories", "")
        primary_category = categories.split()[0]

        # must be primarily genomics, not just cross-listed
        if primary_category != "q-bio.GN":
            continue

        title    = paper.get("title",    "").strip()
        abstract = paper.get("abstract", "").strip()
        searchable = (title + " " + abstract).lower()

        if any(k.lower() in searchable for k in TOPIC_KEYWORDS):
            candidate_papers.append({
                "id":         paper["id"],
                "title":      title,
                "abstract":   abstract,
                "categories": categories,
                "date":       paper.get("update_date", "")
            })

print(f"Found {len(candidate_papers)} candidate papers")

# Sort by date descending — most recent first
candidate_papers.sort(key=lambda p: p["date"], reverse=True)
selected = candidate_papers[:NUM_PAPERS]

print(f"\nSelected {len(selected)} papers:\n")
for i, paper in enumerate(selected, 1):
    print(f"{i:2d}. {paper['date']}  {paper['title']}")

Found 305 candidate papers

Selected 15 papers:

 1. 2026-06-26  scBench-Long: Verifiable Benchmarking of Long-Horizon Single-Cell Biology
 2. 2026-06-25  Privacy-preserving federated tensor decomposition of single-cell immune data: recovering multicellular programs across institutions
 3. 2026-06-25  Stable-Shift: Biologically Structured Prediction of Transcriptional Responses to Unseen Gene Perturbations
 4. 2026-06-18  SciHorizon-GENE: Benchmarking LLM for Life Sciences Inference from Gene Knowledge to Functional Understanding
 5. 2026-06-17  Beyond Independent Genes: Learning Module-Inductive Representations for Single-Cell Gene Perturbation Prediction
 6. 2026-06-17  PyPeakRankR: Reproducible Peak-Level Feature Extraction for Regulatory Element Ranking
 7. 2026-06-15  CisTransCell: Single-Cell Perturbation Prediction via Gene Function, Regulatory Control, and Cellular Context
 8. 2026-06-09  Single-Cell Cross-Modal Transfer by Adversarial Fine-Tuning of Foundation Models
 9. 2026-

In [2]:
import random
import re
import time
import urllib.request
import urllib.error
from pathlib import Path


DOWNLOAD_DELAY = 1.0

pdf_dir = Path("/kaggle/working/pdfs")
pdf_dir.mkdir(parents=True, exist_ok=True)

downloaded = []
failed = []


for idx, paper in enumerate(selected, 1):

    paper_id = paper["id"]
    # Remove version suffix (e.g. v2)
    clean_id = re.sub(r"v\d+$", "", paper_id)
    filename = clean_id.replace("/", "_") + ".pdf"
    pdf_path = pdf_dir / filename
    
    if pdf_path.exists():
        downloaded.append((paper, pdf_path))
        continue

    url = f"https://arxiv.org/pdf/{clean_id}.pdf"

    try:
        urllib.request.urlretrieve(url, pdf_path)
        downloaded.append((paper, pdf_path))
        print(f"[{idx:2d}/{len(selected)}] {paper['title']}")
        time.sleep(DOWNLOAD_DELAY)

    except urllib.error.HTTPError as e:
        print(f"[{idx:3d}] HTTP {e.code}: {clean_id}")
        failed.append(clean_id)

    except Exception as e:

        print(f"[{idx:3d}] Failed {clean_id}: {e}")
        failed.append(clean_id)

print()
print(f"Successfully downloaded : {len(downloaded)}")
print(f"Failed                 : {len(failed)}")

[ 1/15] scBench-Long: Verifiable Benchmarking of Long-Horizon Single-Cell Biology
[ 2/15] Privacy-preserving federated tensor decomposition of single-cell immune data: recovering multicellular programs across institutions
[ 3/15] Stable-Shift: Biologically Structured Prediction of Transcriptional Responses to Unseen Gene Perturbations
[ 4/15] SciHorizon-GENE: Benchmarking LLM for Life Sciences Inference from Gene Knowledge to Functional Understanding
[ 5/15] Beyond Independent Genes: Learning Module-Inductive Representations for Single-Cell Gene Perturbation Prediction
[ 6/15] PyPeakRankR: Reproducible Peak-Level Feature Extraction for Regulatory Element Ranking
[ 7/15] CisTransCell: Single-Cell Perturbation Prediction via Gene Function, Regulatory Control, and Cellular Context
[ 8/15] Single-Cell Cross-Modal Transfer by Adversarial Fine-Tuning of Foundation Models
[ 9/15] Biological Reasoning-Informed Regression for Interpretable Regulatory DNA Activity Prediction
[10/15] Integrating 

In [ ]:
import re
import fitz


def extract_text_from_pdf(pdf_path):
    """
    Extract reasonably clean text from an academic PDF.
    """
    doc = fitz.open(pdf_path)
    pages = []

    for page in doc:
        blocks = page.get_text("blocks")
        blocks.sort(
            key=lambda b: (
                round(b[1] / 20) * 20,
                b[0]
            )
        )
        text = "\n".join(
            b[4].strip()
            for b in blocks
            if b[4].strip()
        )

        if not text:
            continue

        ascii_ratio = (
            sum(c.isascii() for c in text)
            / max(len(text), 1)
        )

        if ascii_ratio < 0.7:
            continue

        if len(text) < 200:
            continue

        pages.append(text)

    doc.close()
    text = "\n\n".join(pages)

    # Collapse repeated whitespace
    text = re.sub(r"[ \t]+", " ", text)

    # Strip figure and table captions
    text = re.sub(r'\bFig(ure)?\.?\s*\d+[^.]*\.', '', text)
    text = re.sub(r'\bTable\s*\d+[^.]*\.', '', text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    patterns = [
        r"\nReferences\b.*",
        r"\nREFERENCE\b.*",
        r"\nBibliography\b.*",
        r"\nAcknowledg(e)?ments\b.*",
        r"\nSupplementary Materials\b.*",
        r"\nSupplementary Information\b.*",
        r"\nAppendix\b.*",
    ]
    
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if m:
            text = text[:m.start()]
            break

    return text.strip()



def chunk_text(
    text,
    target_words=150,
    overlap_words=25,
):
    """
    Paragraph-aware chunking.
    """

    paragraphs = [
        p.strip()
        for p in text.split("\n\n")
        if len(p.strip()) > 40
    ]
    chunks = []
    current = []
    current_words = 0

    for paragraph in paragraphs:
        words = paragraph.split()
        # Giant paragraph
        if len(words) > target_words * 2:
            if current:
                chunks.append(" ".join(current))
                current = []
                current_words = 0
            i = 0
            while i < len(words):
                chunk = words[i:i + target_words]
                chunks.append(" ".join(chunk))
                i += target_words - overlap_words
            continue

        if current_words + len(words) <= target_words:
            current.append(paragraph)
            current_words += len(words)

        else:
            chunks.append("\n\n".join(current))
            overlap = []
            if overlap_words > 0:
                previous_words = " ".join(current).split()
                overlap = previous_words[-overlap_words:]

            current = [" ".join(overlap),paragraph]

            current_words = len(overlap) + len(words)
            # overlap is already counted in words terms, this is correct
            # but recalculate from actual string to be safe
            current_words = len(" ".join(current).split())

    if current:
        chunks.append("\n\n".join(current))

    return [
        c
        for c in chunks
        if len(c.split()) > 80
    ]


all_chunks = []

for paper, pdf_path in downloaded:
    try:
        text = extract_text_from_pdf(pdf_path)
        if len(text) < 1000:
            print(f"Skipping {paper['id']} (too little text)")
            continue

        chunks = chunk_text(
            text,
            target_words=200,
            overlap_words=40,
        )

        for chunk_idx, chunk in enumerate(chunks):
            all_chunks.append({
                "paper_id": paper["id"],
                "chunk_id":
                    f"{paper['id']}_{chunk_idx}",
                "chunk_index": chunk_idx,
                "title": paper["title"],
                "text": chunk
            })

        print(
            f"{paper['id']}: "
            f"{len(chunks)} chunks"
        )

    except Exception as e:
        print(
            f"Error processing "
            f"{paper['id']}: {e}"
        )

print()
print(f"Total chunks: {len(all_chunks):,}")

In [ ]:
from transformers import pipeline
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import torch

import gc
gc.collect()
torch.cuda.empty_cache()

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

from transformers import pipeline, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

generator = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.3",
    model_kwargs={
        "quantization_config": bnb_config,
        "torch_dtype": torch.float16,
    },
    device_map="auto",
)

generator.model.generation_config.max_length = None

print("Generator reloaded in 4-bit.")

In [ ]:
import json
import os
import random
import re

OUTPUT_FILE = "/kaggle/working/instruction_dataset.jsonl"
CHECKPOINT_FILE = "/kaggle/working/checkpoint.txt"

TASK_PROMPTS = {
    "summary":
        "Write an instruction asking for a concise summary of the passage.",

    "explanation":
        "Write an instruction asking for an explanation of the primary biological concept, finding, or result.",

    "definition":
        "Write an instruction asking for the definition or biological role of an important entity mentioned in the passage.",

    "comparison":
        "Write an instruction asking for a comparison between two methods, genes, datasets, organisms, or concepts discussed in the passage. If no meaningful comparison exists, generate an explanation instead.",

    "mechanism":
        "Write an instruction asking for the biological mechanism described in the passage.",

    "cause_effect":
        "Write an instruction asking about a cause-and-effect relationship described in the passage.",

    "methodology":
        "Write an instruction asking for the experimental or computational methodology used.",

    "extraction":
        "Write an instruction asking to extract the important biological entities, genes, proteins, datasets, or findings from the passage.",

    "qa":
        "Write a factual question that can be answered solely from the passage."
}

TASK_POOL = (
    ["summary"] * 4 +
    ["explanation"] * 4 +
    ["mechanism"] * 3 +
    ["methodology"] * 3 +
    ["cause_effect"] * 3 +
    ["comparison"] * 2 +
    ["definition"] * 2 +
    ["extraction"] * 2 +
    ["qa"]
)


def choose_tasks(n=3):
    chosen = []

    while len(chosen) < n:
        task = random.choice(TASK_POOL)
        if task not in chosen:
            chosen.append(task)

    return chosen


def generate_instruction_examples(chunk_text, title, retries=3):

    chosen = choose_tasks()

    task_descriptions = "\n\n".join(
        f"{i+1}. {task}\n{TASK_PROMPTS[task]}"
        for i, task in enumerate(chosen)
    )

    json_template = ",\n".join(
        f'''  {{
    "instruction": "...",
    "response": "...",
    "type": "{task}"
  }}'''
        for task in chosen
    )

    prompt = f"""You are creating instruction tuning data for a biomedical language model.

Generate EXACTLY THREE instruction-response examples.

The examples MUST correspond to the following task types:

{task_descriptions}

Rules:
- Use ONLY information contained in the passage.
- Do NOT invent facts.
- Do NOT reference the paper title, authors, figures, tables, or any metadata.
- Instructions must refer to "the passage" not "the paper" or its title.
- Each instruction should require a different skill.
- The "type" field MUST exactly match the requested task.
- Return ONLY valid JSON.

Paper title:
{title}

Passage:
{chunk_text}

Output:

[
{json_template}
]
"""

    for attempt in range(retries):
        try:
            messages = [{"role": "user", "content": prompt}]

            outputs = generator(
                messages,
                max_new_tokens=768,
                temperature=0.2,
                do_sample=False,
                return_full_text=False,
            )
            
            # Defensively extract text regardless of output structure
            raw = outputs[0]["generated_text"]
            if isinstance(raw, str):
                text = raw.strip()
            elif isinstance(raw, list):
                # list of message dicts
                if isinstance(raw[-1], dict):
                    text = raw[-1].get("content", "").strip()
                else:
                    text = str(raw[-1]).strip()
            elif isinstance(raw, dict):
                text = raw.get("content", "").strip()
            else:
                raise ValueError(f"Unexpected generated_text type: {type(raw)}")
            
            
            generated = outputs[0]["generated_text"]
    
            # chat format returns list of message dicts
            if isinstance(generated, list):
                text = generated[-1]["content"].strip()
            else:
                text = generated.strip()
            text = re.sub(r"^```(?:json)?", "", text).strip()
            text = re.sub(r"```$", "", text).strip()

            start = text.find("[")
            if start == -1:
                raise ValueError("No JSON array found")
            
            text = text[start:]
            
            # Try full parse first
            end = text.rfind("]")
            if end != -1:
                try:
                    examples = json.loads(text[:end+1])
                except json.JSONDecodeError:
                    examples = None
            else:
                examples = None
            
            # Fallback: extract complete objects one by one
            if examples is None:
                examples = []
                for match in re.finditer(r'\{[^{}]+\}', text, re.DOTALL):
                    try:
                        obj = json.loads(match.group())
                        examples.append(obj)
                    except json.JSONDecodeError:
                        continue

            if not isinstance(examples, list):
                raise ValueError("Model did not return a list.")

            valid = []

            for ex in examples:

                if not isinstance(ex, dict):
                    continue

                inst = ex.get("instruction", "").strip()
                resp = ex.get("response", "").strip()
                typ = ex.get("type", "").strip().lower()

                if typ not in chosen:
                    continue

                if len(inst) < 15 or len(resp) < 20:
                    continue

                valid.append({
                    "instruction": inst,
                    "response": resp,
                    "type": typ
                })

            if len(valid) >= 1:
                return valid

            print(f"Only {len(valid)} valid examples returned.")

        except Exception as e:
            print(f"Attempt {attempt+1}: {e}")

    return []

In [ ]:
result = generate_instruction_examples(
    all_chunks[41]["text"],
    all_chunks[41]["title"]
)
print(result)

In [ ]:
start_index = 0

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        start_index = int(f.read()) + 1

dataset = []

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        dataset = [json.loads(line) for line in f]

print(f"Resuming from chunk {start_index}")
print(f"Loaded {len(dataset)} existing examples")

for i, chunk in enumerate(all_chunks):
    if i < start_index:
        continue

    examples = generate_instruction_examples(
        chunk["text"], chunk["title"]
    )

    with open(OUTPUT_FILE, "a") as f:
        for ex in examples:
            record = {
                "paper_id":    chunk["paper_id"],
                "chunk_id":    chunk["chunk_id"],
                "chunk_index": chunk["chunk_index"],
                "title":       chunk["title"],
                "context":     chunk["text"],
                "instruction": ex["instruction"],
                "response":    ex["response"],
                "task_type":   ex["type"],
            }
            dataset.append(record)
            f.write(json.dumps(record) + "\n")

    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(i))

    if (i + 1) % 20 == 0:
        print(f"{i+1}/{len(all_chunks)} chunks | "
              f"{len(dataset)} examples")

print(f"\nFinished. {len(dataset)} instruction-response examples.")

In [4]:
import json

data_path = "/kaggle/input/datasets/tomatopdf/single-cell-qa-dataset/instruction_dataset.jsonl"

with open(data_path) as f:
    dataset = [json.loads(line) for line in f]

print(f"Total examples: {len(dataset)}")
print(f"\nTask type distribution:")

from collections import Counter
task_counts = Counter(d["task_type"] for d in dataset)
for task, count in task_counts.most_common():
    print(f"  {task:15s}: {count}")

print(f"\nSample example:")
print(f"Title:       {dataset[0]['title']}")
print(f"Task:        {dataset[0]['task_type']}")
print(f"Instruction: {dataset[0]['instruction']}")
print(f"Response:    {dataset[0]['response'][:200]}")

Total examples: 1311

Task type distribution:
  explanation    : 214
  summary        : 210
  mechanism      : 182
  methodology    : 161
  cause_effect   : 146
  definition     : 116
  comparison     : 110
  extraction     : 108
  qa             : 64

Sample example:
Title:       SciHorizon-GENE: Benchmarking LLM for Life Sciences Inference from Gene Knowledge to Functional Understanding
Task:        extraction
Instruction: Please extract the number of human genes covered by the SciHorizon-Gene benchmark.
Response:    The SciHorizon-Gene benchmark covers over 190K human genes.


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA devices:", os.environ["CUDA_VISIBLE_DEVICES"])

CUDA devices: 0


In [6]:
import json
import random

random.seed(42)

with open("/kaggle/input/datasets/tomatopdf/single-cell-qa-dataset/instruction_dataset.jsonl") as f:
    dataset = [json.loads(line) for line in f]

random.shuffle(dataset)

split = int(len(dataset) * 0.9)
train_data = dataset[:split]
val_data   = dataset[split:]

def format_example(item):
    return {
        "text": (
            f"<|begin_of_text|>"
            f"<|start_header_id|>system<|end_header_id|>\n"
            f"You are a helpful scientific assistant specializing in "
            f"single-cell transcriptomics and computational genomics. "
            f"Answer questions based only on the provided passage.\n"
            f"<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n"
            f"Passage: {item['context']}\n\n"
            f"{item['instruction']}\n"
            f"<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
            f"{item['response']}"
            f"<|eot_id|>"
        )
    }

train_formatted = [format_example(i) for i in train_data]
val_formatted   = [format_example(i) for i in val_data]

with open("/kaggle/working/train.jsonl", "w") as f:
    for item in train_formatted:
        f.write(json.dumps(item) + "\n")

with open("/kaggle/working/val.jsonl", "w") as f:
    for item in val_formatted:
        f.write(json.dumps(item) + "\n")

print(f"Train: {len(train_formatted)} examples")
print(f"Val:   {len(val_formatted)} examples")
print(f"\nSample formatted text (first 500 chars):")
print(train_formatted[0]["text"][:500])

Train: 1179 examples
Val:   132 examples

Sample formatted text (first 500 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful scientific assistant specializing in single-cell transcriptomics and computational genomics. Answer questions based only on the provided passage.
<|eot_id|><|start_header_id|>user<|end_header_id|>
Passage: KDD 2026, August 9–13, 2026, Jeju Island, Republic of Korea. Yi Duan, Zhao Yang et al. Computational methods offer a promising alternative. If we can accurately predict CRE activity directly from sequence, we can use


In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import gc
gc.collect()
torch.cuda.empty_cache()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count visible: {torch.cuda.device_count()}")
print(f"GPU 0: {torch.cuda.get_device_name(0)}")

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    token=hf_token,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count visible: 1
GPU 0: Tesla T4
Loading tokenizer...
Loading model in 4-bit...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded. Parameters: 1,235,814,400


In [9]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# Force all embedding layers to cuda:0
for name, param in model.named_parameters():
    if "embed" in name:
        param.data = param.data.to("cuda:0")

for name, module in model.named_modules():
    if "embed" in name:
        module.to("cuda:0")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [10]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/working/train.jsonl",
        "validation": "/kaggle/working/val.jsonl"
    }
)

print(f"Train: {len(dataset['train'])}  Val: {len(dataset['validation'])}")

sft_config = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_steps=60,
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train: 1179  Val: 132


Adding EOS to train dataset:   0%|          | 0/1179 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1179 [00:00<?, ? examples/s]

...

Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,2.222983,2.233830,2.269672,182631.000000,0.569071
100,1.894687,1.878933,1.883586,367063.000000,0.608817
150,1.546071,1.598732,1.690379,544296.000000,0.649137
200,1.240626,1.369780,1.374404,719439.000000,0.695298
250,0.993728,1.144807,1.253078,908211.000000,0.741302
300,0.572920,0.846657,0.951880,1087638.000000,0.800355
350,0.402174,0.747285,0.839716,1272687.000000,0.825266
400,0.404566,0.550668,0.696448,1455272.000000,0.872968
450,0.205768,0.453621,0.516023,1633519.000000,0.894329
500,0.169905,0.418966,0.487258,1819099.000000,0.905996


Training complete.
Starting training...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [10]:
import trl 
print(trl.__version__)        

1.7.0


In [11]:
from peft import PeftModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import torch

user_secrets = UserSecretsClient()
hf_write_token = user_secrets.get_secret("HF_WRITE_TOKEN")
login(token=hf_write_token)

# Save the LoRA adapter
adapter_path = "/kaggle/working/lora_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

# Merge LoRA weights back into base model
# Need to reload base model in full precision for merge
print("Reloading base model for merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

merged_path = "/kaggle/working/merged_model"
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

repo_id = "coconutpdf/genomics-llama-1b"

merged_model.push_to_hub(
    repo_id,
    private=False,      # True if you want a private model
)

tokenizer.push_to_hub(repo_id)

print("Model uploaded!")

print(f"Merged model saved to {merged_path}")

Adapter saved to /kaggle/working/lora_adapter
Reloading base model for merge...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to /kaggle/working/merged_model


In [ ]:
# Clone llama.cpp for conversion
!git clone --depth 1 https://github.com/ggml-org/llama.cpp \
    /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt

# Convert to F16 GGUF
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/merged_model \
    --outfile /kaggle/working/genomics-llama-1b-f16.gguf \
    --outtype f16

print("GGUF conversion complete.")
!ls -lh /kaggle/working/genomics-llama-1b-f16.gguf

In [12]:
# Test the merged model directly in transformers
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    device_map="auto",
)

test_passage = """
DNA methylation is an epigenetic mechanism involving the addition of 
a methyl group to the cytosine base of DNA, primarily at CpG dinucleotides. 
This modification plays a crucial role in gene regulation, 
genomic imprinting, and the silencing of transposable elements. 
Aberrant DNA methylation patterns are commonly observed in cancer cells,
where tumor suppressor genes are often hypermethylated and silenced.
"""

test_prompt = (
    f"<|begin_of_text|>"
    f"<|start_header_id|>system<|end_header_id|>\n"
    f"You are a helpful scientific assistant. "
    f"Answer questions based only on the provided passage.\n"
    f"<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n"
    f"Passage: {test_passage}\n\n"
    f"Question: What role does DNA methylation play in cancer?\n"
    f"<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n"
)

result = pipe(test_prompt)
print(result[0]["generated_text"][len(test_prompt):])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


DNA methylation plays a crucial role in cancer by silencing tumor suppressor genes, which are often hypermethylated and silenced in cancer cells.
